# 01 · Seleção de variáveis e clusterização
Leia as specs, os atributos estáticos, o de-para e o JSON de nichos publicados em `data/tratados/spec_analogos/`.
A ordem das variáveis no JSON define a prioridade na seleção. A previsão considera as variáveis dinâmicas dos clusters e dos nichos.


## Configurar caminhos

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data/tratados/spec_analogos'
RUN = ROOT / 'data/tratados/analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Conferir os arquivos de entrada

In [ ]:
import json

from feynmans_legacy.analog_study import (
    KEY,
    cluster_snapshot,
    months,
    select_features,
    validate_key,
)

required = ["terra.parquet", "estaticas.parquet", "coordenadas.parquet",
            "lua.parquet", "marte.parquet", "nichos.json"]
missing = [name for name in required if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError(f"Arquivos ausentes em {DATA}: {', '.join(missing)}")


## Carregar os Parquets e validar as chaves

In [ ]:
terra = pd.read_parquet(DATA / "terra.parquet")
static = pd.read_parquet(DATA / "estaticas.parquet")
coords = pd.read_parquet(DATA / "coordenadas.parquet")

terra["anomes"] = months(terra["anomes"]).dt.strftime("%Y%m").astype(int)
validate_key(terra, KEY, "terra")
validate_key(static, ["location_id"], "estaticas")
validate_key(coords, ["location_id"], "coordenadas")

uncovered = set(terra.location_id) - set(coords.location_id)
if uncovered:
    raise ValueError(f"{len(uncovered)} IDs terrestres sem correspondencia no de-para.")
if not (coords.latitude.between(-90, 90) & coords.longitude.between(-180, 180)).all():
    raise ValueError("Coordenadas ausentes ou fora dos limites geograficos.")


## Identificar as variáveis dos nichos

In [ ]:
niches = json.loads((DATA / "nichos.json").read_text(encoding="utf-8"))
candidates = list(dict.fromkeys(
    item["terra"] for body in niches.values() for niche in body.values() for item in niche
))
reference_month = terra["anomes"].max()
snapshot = terra.loc[terra["anomes"] == reference_month].merge(
    static, on="location_id", how="left", validate="one_to_one"
)


## Selecionar variáveis e agrupar os locais

In [ ]:
chosen, report = select_features(snapshot, candidates)
clusters = cluster_snapshot(snapshot, chosen, n_clusters=3)
targets = [name for name in dict.fromkeys(chosen + candidates) if name in terra.columns]


## Salvar os resultados

In [ ]:
(RUN / "selecao.json").write_text(json.dumps({
    "anomes_cluster": int(reference_month), "variaveis_cluster": chosen, "alvos_previsao": targets
}, indent=2), encoding="utf-8")
for name, frame in {"terra": terra, "estaticas": static, "coordenadas": coords,
                    "clusters": clusters, "selecao": report}.items():
    frame.to_parquet(RUN / f"{name}.parquet", index=False)


## Conferir a seleção e os clusters

In [ ]:
print(report.to_string(index=False))
print(snapshot.join(clusters["cluster"]).groupby("cluster")[chosen].mean())